# COE (Chain of Errors) Prediction
Test individual functions and full pipeline from `revlm.metrics.utils.e_gen`.

In [1]:
import os
import sys
from argparse import Namespace
import torch

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
from revlm.run.edit_utils import find_errors
from revlm.metrics.utils.e_gen import (
    parse_cot_sentences,
    verify_sentence,
    process_sample,
    coe_prediction,
    print_coe_results,
    load_coe,
)

In [2]:
# Config - adjust model_name, dataset_name as needed
args = Namespace(
    config="revlm/config/config.yaml",
    editor="baseline",
    model_name="qwen3",
    dataset_name="aokvqa",
    task="mc",
    split="all",
    n_iter=1,
    ckpt_dir=None,
    task_dir=None,
    edit_dir=None,
    pred_dir=None,
    pred_postedit_dir=None,
    suffix="",
    subsample=0,
    dropout=None,
    pred_by="label_maxprob",
    device=None,
)

config = configure_args(args, config_path=args.config)
config.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
config.subsample = args.subsample
config.cot = True
config.rationale = False
config.overwrite = False

print(f"Model: {config.model.name}")
print(f"Dataset: {config.experiment.dataset_name}")

Task evaluation metrics will be saved to results/te/baseline/Qwen3-VL-8B-Instruct/aokvqa
Edit evaluation metrics will be saved to results/ee/baseline/Qwen3-VL-8B-Instruct/aokvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/aokvqa
Post-edit predictions will be saved to results/pred_postedit/baseline/Qwen3-VL-8B-Instruct/aokvqa
Unified filename to save: mc_all.json
Model: Qwen/Qwen3-VL-8B-Instruct
Dataset: aokvqa


In [3]:
# Load model and error samples
model, edit_ds = find_errors(config)
model.model.eval()

errors = edit_ds.data
print(f"Model: {model.device}")
print(f"Error samples: {len(errors)}")

Step 1 (predictions)
Total samples 18195 loaded from results/pred/Qwen3-VL-8B-Instruct/aokvqa/mc_all.json
getting edits predicted by: label_maxprob

model_old predictions:
{'uid': '1', 'image': 'data/images/aokvqa/train2017/000000299207.jpg', 'question': 'What is the man by the bags awaiting?', 'answer': 'cab', 'rationale': 'A train would not be on the street, he would not have luggage waiting for a delivery, and the skateboarder is there and not paying attention to him so a cab is the only possible answer.', 'cot': 'The image shows a man standing by some bags on the street. A train would not be on the street. The man would not have luggage waiting for a delivery on the street. The skateboarder is present and not paying attention to the man.', 'choices': 'skateboarder; train; delivery; cab', 'idx_choices': '(A) skateboarder\n(B) train\n(C) delivery\n(D) cab', 'idx': 0, 'gold': {'label': 'cab', 'choices': {'str': 'skateboarder; train; delivery; cab', 'ls': ['skateboarder', 'train', 'del

## Test Individual Functions

In [4]:
# Test parse_cot_sentences
ex = errors[0]
cot = ex.get('cot', '')
sentences = parse_cot_sentences(cot)

print(f"COT: {cot}")
print(f"Sentences ({len(sentences)}):")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

COT: The image shows a man standing by some bags on the street. A train would not be on the street. The man would not have luggage waiting for a delivery on the street. The skateboarder is present and not paying attention to the man.
Sentences (4):
  [0] The image shows a man standing by some bags on the street.
  [1] A train would not be on the street.
  [2] The man would not have luggage waiting for a delivery on the street.
  [3] The skateboarder is present and not paying attention to the man.


In [5]:
# Test verify_sentence on first sentence
if sentences:
    flag, p_yes, p_no = verify_sentence(model, ex['image'], sentences[0])
    print(f"Sentence: {sentences[0]}")
    print(f"P(yes)={p_yes:.3f}, P(no)={p_no:.3f}, error={flag}")

Sentence: The image shows a man standing by some bags on the street.
P(yes)=1.000, P(no)=0.000, error=0


In [6]:
# Test process_sample
ex_copy = ex.copy()
ex_copy = process_sample(model, ex_copy)

coe = ex_copy['coe_pred']
print(f"uid: {ex_copy['uid']}")
print(f"Question: {ex_copy['question']}")
print(f"Gold: {ex_copy['gold']['label']}, Pred: {ex_copy['pred']['label_maxprob']}")
print(f"Sentences: {coe['sentences']}")
print(f"Subsets ({len(coe['subsets'])}):")
for sub in coe['subsets']:
    mark = 'x' if sub['error'] else 'v'
    print(f"  [{mark}] {sub['indices']} p_yes={sub['p_yes']:.2f} p_no={sub['p_no']:.2f}")

uid: 1
Question: What is the man by the bags awaiting?
Gold: cab, Pred: train
Sentences: ['The image shows a man standing by some bags on the street.', 'A train would not be on the street.', 'The man would not have luggage waiting for a delivery on the street.', 'The skateboarder is present and not paying attention to the man.']
Subsets (15):
  [v] [0] p_yes=1.00 p_no=0.00
  [v] [1] p_yes=1.00 p_no=0.00
  [x] [2] p_yes=0.11 p_no=0.89
  [x] [3] p_yes=0.01 p_no=0.99
  [v] [0, 1] p_yes=1.00 p_no=0.00
  [v] [0, 2] p_yes=0.78 p_no=0.22
  [x] [0, 3] p_yes=0.00 p_no=1.00
  [v] [1, 2] p_yes=1.00 p_no=0.00
  [v] [1, 3] p_yes=0.87 p_no=0.13
  [x] [2, 3] p_yes=0.02 p_no=0.98
  [v] [0, 1, 2] p_yes=1.00 p_no=0.00
  [x] [0, 1, 3] p_yes=0.01 p_no=0.99
  [x] [0, 2, 3] p_yes=0.00 p_no=1.00
  [v] [1, 2, 3] p_yes=1.00 p_no=0.00
  [x] [0, 1, 2, 3] p_yes=0.13 p_no=0.87


## Run Full Pipeline

In [7]:
# # Run COE prediction on all errors and save
# results = coe_prediction(model, edit_ds, config)
# # Inspect results
# print_coe_results(results, max_print=10)

## Load Saved Results & Calculate COE Rate


In [8]:
# Load saved COE results and print rate (uses config.pred_postedit_dir)
results, coe_rate = load_coe(config)

Loaded 7139 samples from results/pred_postedit/baseline/Qwen3-VL-8B-Instruct/aokvqa/coe_prediction.json
COE rate: 3195/7139 (44.8%)

=== uid: 1 ===
Q: What is the man by the bags awaiting?
Gold: cab, Pred: train
Sentences: ['The image shows a man standing by some bags on the street.', 'A train would not be on the street.', 'The man would not have luggage waiting for a delivery on the street.', 'The skateboarder is present and not paying attention to the man.']
  [√] [0] p_yes=1.00 p_no=0.00
  [√] [1] p_yes=1.00 p_no=0.00
  [x] [2] p_yes=0.11 p_no=0.89
  [x] [3] p_yes=0.01 p_no=0.99
  [√] [0, 1] p_yes=1.00 p_no=0.00
  [√] [0, 2] p_yes=0.78 p_no=0.22
  [x] [0, 3] p_yes=0.00 p_no=1.00
  [√] [1, 2] p_yes=1.00 p_no=0.00
  [√] [1, 3] p_yes=0.87 p_no=0.13
  [x] [2, 3] p_yes=0.02 p_no=0.98
  [√] [0, 1, 2] p_yes=1.00 p_no=0.00
  [x] [0, 1, 3] p_yes=0.01 p_no=0.99
  [x] [0, 2, 3] p_yes=0.00 p_no=1.00
  [√] [1, 2, 3] p_yes=1.00 p_no=0.00
  [x] [0, 1, 2, 3] p_yes=0.13 p_no=0.87

=== uid: 8 ===
Q: 